

# Crossvalidation and hyperparameter selection 


In this project we are going to explore the cross validation testing methodology and applying that to get error estimates on the two algorithms:
  - Linear Regression
  - Decision Tree classification
  

In [2]:
# importing libraries
import pandas as pd
import numpy as np
from sklearn import tree 
from sklearn.datasets import load_iris
from sklearn.datasets import load_diabetes

# First we will Calculate Generalized Error on Linear Regression with k-fold Cross Validation

## Loading in the diabetes data set as a pandas dataframe and series.  
The features are the dataframe `df_X` and the target values are the series `s_y`

In [3]:
df = load_diabetes(as_frame=True)
df_X = df.data
s_y = df.target

## Defining a function that creates a linear least squares regression model 
This function takes in two parameters, `df_X`, and `s_y` and return the least squares regression model, $\hat{\beta}$

In [4]:
def get_linear_regression_model( df_X, s_y ):
    df_X = pd.concat([pd.DataFrame({'intercept':np.ones(len(df_X))}), df_X], axis = 1)
    (beta_hat, residuals, rank, s) = np.linalg.lstsq(df_X, s_y, rcond=-1)
    
    return beta_hat

In [5]:
# code to check beta_hat
np.random.seed(23)
beta_hat = get_linear_regression_model( pd.DataFrame(np.random.random((34,4))), pd.Series(np.random.random(34)*10.0) )
beta_hat

array([ 4.18818425,  1.77890808,  0.74032569, -1.3506416 ,  0.14535984])

##  Defining a function that partitions the dataframe and series data into dictionaries
This function takes in three parameters, `df_X`, `s_y`, and `k`, and returns a tuple of two dictionaries.
In both dictionaries the key is the `k` value.
The first dictionary will have the dataframe of the training data that contains approximately $\frac{1}{k}$ of the data.
The second dictionary will have the series of the target data that contains approximately $\frac{1}{k}$ of the data.


In [5]:
def partition_data( df_X, s_y, k):

    dict_k_df_X = {fold_number: pd.Series([0, 0, 0, 0]) for fold_number in range(1,k+1)} 
    dict_k_s_y = {fold_number: pd.Series([0, 0, 0, 0]) for fold_number in range(1,k+1)}
    
    for index, rows in df_X.iterrows():
        
        fold_number = np.random.randint(1,k+1)
        if dict_k_df_X[fold_number].equals(pd.Series([0, 0, 0, 0])):
            dict_k_df_X[fold_number] = rows
            dict_k_s_y[fold_number] = pd.Series([s_y[index]])
        
        else:
            dict_k_df_X[fold_number] = pd.concat([dict_k_df_X[fold_number], rows], axis = 1)
            dict_k_s_y[fold_number] = pd.concat([dict_k_s_y[fold_number], pd.Series([s_y[index]])], axis = 0)
            
    
    for i in range(1,k+1):
        dict_k_df_X[i] = dict_k_df_X[i].transpose()
        dict_k_s_y[i] = dict_k_s_y[i].drop(columns = 'index')
        
    return dict_k_df_X, dict_k_s_y

Calling partition_data function with $k=5$ and creating the dictionaries `dict_k_df_X` and `dict_k_s_y`. 

In [6]:
(dict_k_df_X, dict_k_s_y) = partition_data( df_X, s_y, 5 )

Printing out the number of rows in each fold and Checking that the number of data points in each partition totals the number of data points in the entire dataset. 

In [7]:
# Check fold sizes
total_folds = 0
k = 5
for i in range(1,k+1):
    print('Fold ' + str(i) + ' length of dataframe is ' + str(len(dict_k_df_X[i].index)) + ' and length of series is ' + str(len(dict_k_df_X[i].index)))
    total_folds += len(dict_k_df_X[i].index)
    
print('The sum of the number of elements in each fold is ' + str(total_folds) + ' and there are ' + str(len(df_X.index)) + ' rows in the original df')

Fold 1 length of dataframe is 92 and length of series is 92
Fold 2 length of dataframe is 98 and length of series is 98
Fold 3 length of dataframe is 75 and length of series is 75
Fold 4 length of dataframe is 75 and length of series is 75
Fold 5 length of dataframe is 102 and length of series is 102
The sum of the number of elements in each fold is 442 and there are 442 rows in the original df


## Defining a function that calculates a regression metric
This function accepts two series of equal length $n$ numpy arrays, `s_y`, and `s_y_hat`. The metric it calculates is the mean absolute error, $MAE = \sum\limits_{i=1}^n\frac{|{s\_y_i - {s\_y\_hat}_i}|}{n}$ 


In [8]:
def get_mae( s_y, s_y_hat):
    n = s_y.size
    mae = 0
    for i in range(n):
        mae += abs(s_y[i] - s_y_hat[i]) / n
    
    return mae

Testing the function by using the vectors:
```
x = np.array([1,2,3])
y = np.array([2,2,3])
```

In [9]:
# Test it 
x = np.array([1,2,3])
y = np.array([2,2,3])
get_mae(x,y)

0.3333333333333333

## Calculating the $MAE$ for each fold
For each fold in dictionaries, we will calculate the $MAE$, Using the partition number key as the test set, and all other partitions as the train set. 


In [10]:
mae = np.array([])

train_lst = []
for k in dict_k_df_X.keys():
    
    #Concat training arrays
    train_X = pd.DataFrame(np.concatenate([v for (n, v) in dict_k_df_X.items() if n != k]))
    train_y = pd.DataFrame(np.concatenate([v for (n, v) in dict_k_s_y.items() if n != k]))
    
    # Get beta_hat from training arrays
    beta_hat = get_linear_regression_model(train_X, train_y)
    
    # Add intercept col to test_X
    test_X = dict_k_df_X[k].reset_index(drop = True)
    test_X = pd.DataFrame({'intercept': np.ones(len(test_X))}).join(test_X)
    
    # Get s_y_hat from test set
    s_y_hat = np.dot(np.array(test_X), np.array(beta_hat))
    
    # Calc mae using s_y (with reset index) and s_y_hat
    s_y = np.array(dict_k_s_y[k].reset_index(drop = True))
    mae = np.append(mae, get_mae(s_y, s_y_hat))

Printing the min, max, and mean $MAE$ of the 5 folds. 

In [11]:
print("The min MAE is {:.2f}, the max MAE is {:.2f}, and the mean MAE is {:.2f}".format(mae.min(),mae.max(),mae.mean()))

The min MAE is 40.76, the max MAE is 47.96, and the mean MAE is 44.22


# Finding the best hyperparameter to use in a Decision Tree 

## Loading the iris data in as a pandas dataframe and a series
 With features dataframe `df_X` and class label series `s_y`.

In [12]:
iris = load_iris(as_frame=True)
df_X = iris.data
s_y = iris.target
df_X

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


## Partitioning `df_X` and `s_y` into $5$ partitions of roughly equal size
We will make 2 dictionaries, with the key of each dictionary the fold number.  The value of the dictionary `dict_k_df_X` is the $k^{th}$ partition of the data, and the value of the dictionary `dict_k_s_y` is the corresponding $k^{th}$ target classification.

In [13]:
(dict_k_df_X, dict_k_s_y) = partition_data( df_X, s_y, 5 )

## Defining a function that calculates accuracy
This function accepts two series and compare each element for equality. The accuracy is the number of equal elements divided by the total number of elements.


In [14]:
def get_acc( s_1, s_2 ):
    s_1 = s_1.to_numpy()
    accuracy = (s_1 == s_2).sum()/len(s_1)
    
    return accuracy

Testing the accuracy function by calling it with the `s_y` loaded from the iris data set and an array of the same length containing all $1$ values. 

In [15]:
get_acc(s_y,np.ones(len(s_y)))

0.3333333333333333

## Using Nested Cross validation, finding the best hyperparameter
Using the [Decision Tree Classifier](https://scikit-learn.org/stable/modules/tree.html#classification) class to build a decision tree inside of a 5-fold cross validation loop. The partitions we created in the previous parts will be the outer loop.  The inside loop uses 4-fold cross validation to find the best value for `min_impurity_decrease`. We will use the Gini Index as the impurity measure. 
   Then calculating the mean accuracy across the 4 folds of the inner loop for all the candidate `min_impurity_decrease` values, and print the value. 
   Then we will use the array `np.array([0.1,0.25,0.3,0.4])` as the candidates for the best hyperparameter. If there is a tie  we'll choose the lowest `min_impurity_decrease` value. 

For each inner loop, we select the best `min_impurity_decrease` and train the outer fold training data on using that value. 


In [16]:
possible_min_impurity_decrease = np.array([0.1,0.25,0.3,0.4])

# Outer loop
outer_acc = np.array([])
for k in dict_k_df_X.keys():
    print('Fold ' + str(k))
    
    #test 
    test_X = dict_k_df_X[k]
    test_Y = dict_k_s_y[k]
    idx = test_X.index
    test_X = test_X.reset_index(drop=True)
    test_Y = test_Y.reset_index(drop=True)
    
    #train
    train_X = df_X.drop(idx)
    train_Y = s_y.drop(idx)
    
    #make the partitions
    (inner_dict_X, inner_dict_y) = partition_data( train_X, train_Y, 4 )
    
    inner_acc = np.array([])
    best = 0
    for pos_min_impurity in possible_min_impurity_decrease:
        
        # Inner loop cross validation
        for i in inner_dict_X.keys():
            inner_test_X = inner_dict_X[i]
            inner_test_Y = inner_dict_y[i]
            idx = inner_test_X.index
            inner_test_X = inner_test_X.reset_index(drop = True)
            inner_test_Y = inner_test_Y.reset_index(drop = True)
            
            inner_train_X = train_X.drop(idx)
            inner_train_Y = train_Y.drop(idx)
            
            clf = tree.DecisionTreeClassifier(min_impurity_decrease = pos_min_impurity)
            clf.fit(inner_train_X, inner_train_Y)
            s_y_hat = clf.predict(inner_test_X)
            
            acc = get_acc(inner_test_Y, s_y_hat)
            inner_acc = np.append(inner_acc,acc)
        
        avg = inner_acc.mean()
        print('Testing {} min impurity decrease. Average accuracy over 4 folds is {:.2f}'.format(pos_min_impurity, avg))
        
        if avg > best:
            best = avg
            parameter = pos_min_impurity
        
    print('Best min impurity decrease is '+ str(parameter))
    print()
    # Use best min impurity decrease to train model
    clf = tree.DecisionTreeClassifier(min_impurity_decrease = parameter)
    clf.fit(train_X, train_Y)
    
    # outer accuracy calculation 
    s_y_hat = clf.predict(test_X)
    this_acc = get_acc(test_Y, s_y_hat)

    outer_acc = np.append(outer_acc,this_acc)

Fold 1
Testing 0.1 min impurity decrease. Average accuracy over 4 folds is 0.93
Testing 0.25 min impurity decrease. Average accuracy over 4 folds is 0.88
Testing 0.3 min impurity decrease. Average accuracy over 4 folds is 0.76
Testing 0.4 min impurity decrease. Average accuracy over 4 folds is 0.63
Best min impurity decrease is 0.1

Fold 2
Testing 0.1 min impurity decrease. Average accuracy over 4 folds is 0.91
Testing 0.25 min impurity decrease. Average accuracy over 4 folds is 0.86
Testing 0.3 min impurity decrease. Average accuracy over 4 folds is 0.80
Testing 0.4 min impurity decrease. Average accuracy over 4 folds is 0.67
Best min impurity decrease is 0.1

Fold 3
Testing 0.1 min impurity decrease. Average accuracy over 4 folds is 0.97
Testing 0.25 min impurity decrease. Average accuracy over 4 folds is 0.97
Testing 0.3 min impurity decrease. Average accuracy over 4 folds is 0.86
Testing 0.4 min impurity decrease. Average accuracy over 4 folds is 0.71
Best min impurity decrease is 

## Showing the generalized performance of the classifier 
Showing the generalized performance of the classifier by printing the min, max, and mean accuracy of the outer fold test sets.

In [17]:
print("The min accuracy is {:.2f}, the max accuracy is {:.2f}, and the mean accuracy is {:.2f}".format(outer_acc.min(),outer_acc.max(),outer_acc.mean()))

The min accuracy is 0.89, the max accuracy is 0.97, and the mean accuracy is 0.94
